In [1]:
import pathlib
import warnings
from functools import reduce

import pandas as pd

warnings.filterwarnings("ignore")  # Ignore all warnings
warnings.simplefilter("ignore")  # Additional suppression method

from notebook_init_utils import init_notebook

root_dir, in_notebook = init_notebook()

In [2]:
profile_dict = {
    "organoid_fs": {
        "input_profile_path": pathlib.Path(
            root_dir, "1.EDA/results/linear_modeling/organoid_fs.parquet"
        ),
    },
    "single_cell_fs": {
        "input_profile_path": pathlib.Path(
            root_dir, "1.EDA/results/linear_modeling/sc_fs.parquet"
        ),
    },
}

count_covariates = ["cell_count", "organoid_count", "cell_per_organoid_count"]

thresholds_dict = {
    "organoid_fs": {
        "pvalue_max": 0.05,  # significance threshold for p-values
        "rsquared_min": 0.4,  # explained variance floor
        "rsquared_adj_min": 0,  # the model performs better than the null model
        "coefficient_min": 1,  # minimum effect size, treatment/patient terms only
    },
    "single_cell_fs": {
        "pvalue_max": 0.05,
        "rsquared_min": 0.5,
        "rsquared_adj_min": 0,
        "coefficient_min": 1,
    },
}

## Filter significant features

The linear model (see `9.linear_modeling.ipynb`) is fit per (treatment combo, feature) and
produces one row per **term**: `treatment`, `patient`, `cell_count`, `organoid_count`, and
`cell_per_organoid_count`.

- pvalue threshold: statistically significant features
- rsquared / rsquared adjusted thresholds: the model explains a meaningful share of variance
- coefficient threshold: minimum effect size, applied only to the `treatment` and `patient`
  terms (both are categorical dummy shifts on the same scale). The count covariates are on a
  per-unit scale, so they are filtered on significance/fit only, not coefficient magnitude.

**Analysis axes:**

- **Treatment**: which features are significant for each treatment; union/intersection across treatments
- **Patient**: which features are treatment-responsive in each patient; union/intersection across patients
- **Therapeutic category (MOA)**: treatments grouped by `therapeutic_category`; union/intersection across categories
- **Covariates**: which features are significantly explained by cell/organoid count terms; their
  overlap with the treatment/category sets flags features whose apparent "treatment effect" may be
  confounded by density rather than a true biological effect

In [3]:
def significant_feature_sets(
    term_df, group_col, thresholds, use_coefficient_threshold=True
):
    """Return {group_value: set(features)} for rows that pass the thresholds."""
    mask = (
        (term_df["pvalue"] < thresholds["pvalue_max"])
        & (term_df["rsquared"] > thresholds["rsquared_min"])
        & (term_df["rsquared_adj"] > thresholds["rsquared_adj_min"])
    )
    if use_coefficient_threshold:
        mask &= term_df["coefficient"].abs() > thresholds["coefficient_min"]
    sig_df = term_df[mask]
    return {
        group_value: set(group_df["feature"])
        for group_value, group_df in sig_df.groupby(group_col)
    }


def covariate_significant_set(df, covariate, thresholds):
    """Features where `covariate` is a significant predictor in at least one treatment combo."""
    term_df = df[df["term"] == covariate]
    mask = (
        (term_df["pvalue"] < thresholds["pvalue_max"])
        & (term_df["rsquared"] > thresholds["rsquared_min"])
        & (term_df["rsquared_adj"] > thresholds["rsquared_adj_min"])
    )
    return set(term_df.loc[mask, "feature"])


def union_and_intersection(sets_dict):
    if not sets_dict:
        return set(), set()
    sets = list(sets_dict.values())
    return reduce(set.union, sets), reduce(set.intersection, sets)

In [4]:
results_by_profile = {}
for profile_name, profile_info in profile_dict.items():
    print(f"\n{'=' * 20} {profile_name} {'=' * 20}")
    df = pd.read_parquet(profile_info["input_profile_path"])
    thresholds = thresholds_dict[profile_name]

    treatment_term_df = df[df["term"] == "treatment"]
    patient_term_df = df[df["term"] == "patient"]

    treatment_sets = significant_feature_sets(
        treatment_term_df, "treatment", thresholds
    )
    patient_response_sets = significant_feature_sets(
        treatment_term_df, "patient", thresholds
    )
    patient_baseline_sets = significant_feature_sets(
        patient_term_df, "patient", thresholds
    )
    category_sets = significant_feature_sets(
        treatment_term_df, "therapeutic_category", thresholds
    )
    covariate_sets = {
        covariate: covariate_significant_set(df, covariate, thresholds)
        for covariate in count_covariates
    }

    treatment_union, treatment_intersection = union_and_intersection(treatment_sets)
    patient_response_union, patient_response_intersection = union_and_intersection(
        patient_response_sets
    )
    patient_baseline_union, patient_baseline_intersection = union_and_intersection(
        patient_baseline_sets
    )
    category_union, category_intersection = union_and_intersection(category_sets)

    print(
        f"treatment: {len(treatment_sets)} treatments, "
        f"union={len(treatment_union)}, intersection={len(treatment_intersection)}"
    )
    print(
        f"patient (treatment-responsive): {len(patient_response_sets)} patients, "
        f"union={len(patient_response_union)}, intersection={len(patient_response_intersection)}"
    )
    print(
        f"patient (baseline shift): {len(patient_baseline_sets)} patients, "
        f"union={len(patient_baseline_union)}, intersection={len(patient_baseline_intersection)}"
    )
    print(
        f"therapeutic_category: {len(category_sets)} categories, "
        f"union={len(category_union)}, intersection={len(category_intersection)}"
    )
    for covariate, feats in covariate_sets.items():
        print(f"  {covariate}: {len(feats)} significant features")

    # how much of each treatment's significant-feature set overlaps with each
    # count covariate's significant-feature set (possible confounding)
    overlap_rows = []
    for treatment, feats in treatment_sets.items():
        row = {"treatment": treatment, "n_significant_features": len(feats)}
        for covariate, cov_feats in covariate_sets.items():
            overlap = feats & cov_feats
            row[f"{covariate}_overlap_n"] = len(overlap)
            row[f"{covariate}_overlap_frac"] = (
                len(overlap) / len(feats) if feats else 0.0
            )
        overlap_rows.append(row)
    overlap_df = pd.DataFrame(overlap_rows).sort_values(
        "n_significant_features", ascending=False
    )

    results_by_profile[profile_name] = {
        "treatment_sets": treatment_sets,
        "patient_response_sets": patient_response_sets,
        "patient_baseline_sets": patient_baseline_sets,
        "category_sets": category_sets,
        "covariate_sets": covariate_sets,
        "treatment_union": treatment_union,
        "treatment_intersection": treatment_intersection,
        "patient_response_union": patient_response_union,
        "patient_response_intersection": patient_response_intersection,
        "patient_baseline_union": patient_baseline_union,
        "patient_baseline_intersection": patient_baseline_intersection,
        "category_union": category_union,
        "category_intersection": category_intersection,
        "covariate_overlap_df": overlap_df,
    }
    print(overlap_df.to_string(index=False))


==================== organoid_fs ====================


KeyError: 'term'